# Speculative Decoding, Derived: Draft Models, N-grams, Medusa and EAGLE

The full code of [Speculative Decoding, Derived: Draft Models, N-grams, Medusa and EAGLE](https://residual-stream.lvtan71.workers.dev/posts/speculative-decoding/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/speculative-decoding) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/speculative-decoding.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 38 min end to end on the RTX 3090 machine that saved the outputs below.
Downloads Qwen2.5-1.5B-Instruct and Qwen2.5-0.5B, about 4 GB. The target's text differs by machine, so acceptance rates move; prompt lookup gave 2.5 tokens per step on an RTX 3090 against 2.9 in the post.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-17** (torch 2.14.1+cu130, 37 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "numpy==2.*" "torch==2.14.*" "transformers==5.17.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "speculative-decoding"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/speculative-decoding


## `spec_math.py`

Checks by sampling that the acceptance rule reproduces the target's distribution with a bad draft; tokens per verification step, formula against simulation; speedup by draft cost.

In [2]:
!{sys.executable} spec_math.py

target p:           [0.4  0.25 0.15 0.1  0.07 0.03]
output frequencies: [0.4  0.25 0.15 0.1  0.07 0.03]
acceptance rate 0.550; theory: sum of min(p, q) = 0.550

alpha  k   formula  simulated
  0.6  2      1.96       1.96


  0.6  4      2.31       2.31
  0.6  8      2.47       2.48
  0.8  2      2.44       2.44
  0.8  4      3.36       3.36
  0.8  8      4.33       4.32


  0.9  2      2.71       2.71
  0.9  4      4.10       4.10
  0.9  8      6.13       6.13

speedup over plain decoding, by draft cost c (alpha = 0.8)
  k   c=0      c=0.05   c=0.1    c=0.3  
  1     1.80    1.71    1.64    1.38
  2     2.44    2.22    2.03    1.52
  4     3.36    2.80    2.40    1.53
  8     4.33    3.09    2.40    1.27
 16     4.89    2.72    1.88    0.84


The post printed:

```text
target p:           [0.4  0.25 0.15 0.1  0.07 0.03]
output frequencies: [0.4  0.25 0.15 0.1  0.07 0.03]
acceptance rate 0.550; theory: sum of min(p, q) = 0.550

alpha  k   formula  simulated
  0.6  2      1.96       1.96
  0.6  4      2.31       2.31
  0.6  8      2.47       2.48
  0.8  2      2.44       2.44
  0.8  4      3.36       3.36
  0.8  8      4.33       4.32
  0.9  2      2.71       2.71
  0.9  4      4.10       4.10
  0.9  8      6.13       6.13

speedup over plain decoding, by draft cost c (alpha = 0.8)
  k   c=0      c=0.05   c=0.1    c=0.3  
  1     1.80    1.71    1.64    1.38
  2     2.44    2.22    2.03    1.52
  4     3.36    2.80    2.40    1.53
  8     4.33    3.09    2.40    1.27
 16     4.89    2.72    1.88    0.84
```

## `accept.py`

Qwen2.5-1.5B-Instruct as the target, Qwen2.5-0.5B as the draft: per-position greedy agreement, sum of min(p, q), and two-token prompt lookup, on paragraph rewrites and open questions.

In [3]:
!{sys.executable} accept.py

rewrite a paragraph  (666 tokens): draft model, greedy 80%, sampled 75%; prompt lookup 42%


open question        (691 tokens): draft model, greedy 73%, sampled 72%; prompt lookup 7%


The post printed:

```text
rewrite a paragraph  (666 tokens): draft model, greedy 79%, sampled 75%; prompt lookup 42%
open question        (720 tokens): draft model, greedy 72%, sampled 70%; prompt lookup 6%
```

## `spec_steps.py`

The same models and prompts, measured as speculation runs: tokens per step for greedy speculation at k = 4 and 8 against the formula, acceptance on sampled text, and prompt lookup as deployed.

In [4]:
!{sys.executable} spec_steps.py

vocab 151936 151936


{'n': np.int64(93), 'g': np.float64(0.731), 'smin_g': np.float64(0.681), 'smin_s': np.float64(0.644), 'ns': np.int64(118), 'lk1': np.float64(0.215), 'spec4': np.float64(2.906), 'spec8': np.float64(3.577), 'pl': (np.float64(1.453), np.float64(0.656), np.float64(0.286)), 'pl2': (np.float64(1.453), np.float64(0.656), np.float64(0.286))}


{'n': np.int64(115), 'g': np.float64(0.774), 'smin_g': np.float64(0.697), 'smin_s': np.float64(0.587), 'ns': np.int64(114), 'lk1': np.float64(0.374), 'spec4': np.float64(3.026), 'spec8': np.float64(3.966), 'pl': (np.float64(1.855), np.float64(0.613), np.float64(0.395)), 'pl2': (np.float64(1.855), np.float64(0.613), np.float64(0.395))}


{'n': np.int64(120), 'g': np.float64(0.95), 'smin_g': np.float64(0.886), 'smin_s': np.float64(0.604), 'ns': np.int64(120), 'lk1': np.float64(0.725), 'spec4': np.float64(4.286), 'spec8': np.float64(7.059), 'pl': (np.float64(5.714), np.float64(0.762), np.float64(0.812)), 'pl2': (np.float64(5.217), np.float64(0.783), np.float64(0.722))}


{'n': np.int64(98), 'g': np.float64(0.673), 'smin_g': np.float64(0.647), 'smin_s': np.float64(0.639), 'ns': np.int64(100), 'lk1': np.float64(0.194), 'spec4': np.float64(2.722), 'spec8': np.float64(2.97), 'pl': (np.float64(1.485), np.float64(0.606), np.float64(0.45)), 'pl2': (np.float64(1.485), np.float64(0.606), np.float64(0.45))}


...

{'n': np.int64(91), 'g': np.float64(0.692), 'smin_g': np.float64(0.691), 'smin_s': np.float64(0.726), 'ns': np.int64(120), 'lk1': np.float64(0.066), 'spec4': np.float64(2.758), 'spec8': np.float64(3.138), 'pl': (np.float64(1.123), np.float64(0.395), np.float64(0.156)), 'pl2': (np.float64(1.123), np.float64(0.395), np.float64(0.156))}


{'n': np.int64(120), 'g': np.float64(0.675), 'smin_g': np.float64(0.702), 'smin_s': np.float64(0.707), 'ns': np.int64(120), 'lk1': np.float64(0.083), 'spec4': np.float64(2.609), 'spec8': np.float64(2.927), 'pl': (np.float64(1.176), np.float64(0.451), np.float64(0.174)), 'pl2': (np.float64(1.176), np.float64(0.451), np.float64(0.174))}


{'n': np.int64(120), 'g': np.float64(0.75), 'smin_g': np.float64(0.743), 'smin_s': np.float64(0.715), 'ns': np.int64(120), 'lk1': np.float64(0.025), 'spec4': np.float64(3.077), 'spec8': np.float64(3.636), 'pl': (np.float64(1.091), np.float64(0.318), np.float64(0.2)), 'pl2': (np.float64(1.091), np.float64(0.318), np.float64(0.2))}


{'n': np.int64(120), 'g': np.float64(0.75), 'smin_g': np.float64(0.737), 'smin_s': np.float64(0.723), 'ns': np.int64(120), 'lk1': np.float64(0.075), 'spec4': np.float64(3.0), 'spec8': np.float64(3.636), 'pl': (np.float64(1.165), np.float64(0.379), np.float64(0.256)), 'pl2': (np.float64(1.165), np.float64(0.379), np.float64(0.256))}


== open: n=691; greedy agree 0.732 (per-prompt min 0.68 max 0.76)
   smin on greedy text 0.723; smin on T=1 sampled text 0.705
   lookup 1-token 2-gram 0.068
   greedy spec tokens/step k=4: actual 2.91, iid formula 2.95; k=8 actual 3.49, formula 3.51
   prompt lookup pl (tokens/step, frac steps with a proposal, frac proposals with >=1 accepted): (np.float64(1.142), np.float64(0.417), np.float64(0.183))
   prompt lookup pl2 (tokens/step, frac steps with a proposal, frac proposals with >=1 accepted): (np.float64(1.142), np.float64(0.417), np.float64(0.183))


The post printed:

```text
== rewrite: n=666; greedy agree 0.787 (per-prompt min 0.58 max 0.99)
   smin on greedy text 0.754; smin on T=1 sampled text 0.715
   lookup 1-token 2-gram 0.422
   greedy spec tokens/step k=4: actual 3.27, iid formula 3.28; k=8 actual 4.32, formula 4.15
   prompt lookup pl (tokens/step, frac steps with a proposal, frac proposals with >=1 accepted): (np.float64(2.937), np.float64(0.715), np.float64(0.456))
== open: n=720; greedy agree 0.718 (per-prompt min 0.68 max 0.76)
   smin on greedy text 0.702; smin on T=1 sampled text 0.722
   lookup 1-token 2-gram 0.060
   greedy spec tokens/step k=4: actual 2.82, iid formula 2.87; k=8 actual 3.34, formula 3.37
   prompt lookup pl (tokens/step, frac steps with a proposal, frac proposals with >=1 accepted): (np.float64(1.125), np.float64(0.403), np.float64(0.179))
```